# Avro Basics — 05: Nullable Unions


Avro represents nullable fields as unions with null.
This is the most common pattern you'll encounter in Avro schemas.

Topics: ["null","T"] pattern, default values for unions, complex unions,
reading legacy Avro with nullable fields, null handling in Spark.


In [ ]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

## Step 1 — The Nullable Union Pattern

Nullable fields use union with null. Order matters for the default.

In [ ]:

import json as pyjson

# WRONG: no default — required field
# CORRECT nullable patterns
print("Nullable union patterns:")
print()
print('  ["null","string"]  — nullable, default null:')
print('  {"type":["null","string"],"default":null}')
print()
print('  ["string","null"]  — nullable, default "" (string first):')
print('  {"type":["string","null"],"default":""}')
print()
print("Rule: the FIRST type in the union must match the default value type")

NULLABLE_SCHEMA = pyjson.dumps({"type":"record","name":"Customer","fields":[
    {"name":"id",          "type":"long"},
    {"name":"name",        "type":"string"},
    {"name":"email",       "type":["null","string"],  "default":None},   # nullable, default null
    {"name":"phone",       "type":["string","null"],  "default":""},     # nullable, default ""
    {"name":"score",       "type":["null","double"],  "default":None},   # nullable double
    {"name":"is_vip",      "type":["null","boolean"], "default":None},   # nullable boolean
]})
print("\nSchema with nullable fields created ✅")


## Step 2 — Write and Read Nullable Data



In [ ]:

customers = spark.createDataFrame([
    (1, "Alice", "alice@ex.com",  "+1-555-0101", 9.5,  True),
    (2, "Bob",   None,             "",            None, False),
    (3, "Carol", "carol@ex.com",  None,           7.2,  None),
], ["id","name","email","phone","score","is_vip"])

CUST_PATH = f"{DATA_DIR}/nullable_customers"
customers.write.format("avro").option("avroSchema", NULLABLE_SCHEMA) \
         .mode("overwrite").save(CUST_PATH)

df_cust = spark.read.format("avro").load(CUST_PATH)
print("Nullable fields in Avro:")
df_cust.printSchema()
print()
df_cust.show(truncate=False)
print("null values preserved correctly ✅")


## Step 3 — Handling Nulls in Queries



In [ ]:

df_cust = spark.read.format("avro").load(f"{DATA_DIR}/nullable_customers")

print("Null count per column:")
df_cust.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_cust.columns]).show()

print("Fill nulls with defaults:")
df_filled = df_cust \
    .fillna({"email": "unknown@example.com", "phone": "N/A", "score": 0.0}) \
    .fillna({"is_vip": False})
df_filled.show(truncate=False)


## Step 4 — Complex Unions (Multiple Non-Null Types)



In [ ]:
# Complex union: value can be int OR string OR null
# ["null","string","long"] is valid Avro but Spark CANNOT write it directly
# — Spark maps each SQL type to exactly ONE Avro type, not a multi-type union.

print('Complex union limitation in Spark:')
print()
print('  Avro schema: metadata = ["null", "string", "long"]')
print('  Spark SQL  : metadata = STRING (or LONG — only one type possible)')
print()
print('Spark can READ multi-type unions (resolves to widest compatible type)')
print('Spark CANNOT WRITE multi-type unions — IncompatibleSchemaException')
print()
print('Root cause:')
print('  Spark serializer maps each DataFrame column to exactly one Avro type.')
print('  A union of [null, string, long] has 2 non-null options — ambiguous.')
print()
print('Correct patterns instead of complex unions:')
print()
print('  Pattern 1: separate typed fields (recommended)')
print('  {name: metadata_str,  type: ["null","string"], default: null}')
print('  {name: metadata_long, type: ["null","long"],   default: null}')
print()
print('  Pattern 2: serialize value as JSON string')
print('  {name: metadata, type: ["null","string"], default: null}')
print('  Store numbers as: {"type": "count", "value": 42}')
print()

# Pattern 2 demo: JSON string workaround — works correctly
import json as pyjson
SAFE_SCHEMA = pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id", "type":"long"},
    {"name":"metadata", "type":["null","string"], "default":None},
]})

from pyspark.sql.types import StructType, StructField, LongType, StringType
from pyspark.sql import Row

safe_data = spark.createDataFrame([
    Row(event_id=1, metadata=None),
    Row(event_id=2, metadata="campaign_abc"),
    Row(event_id=3, metadata=pyjson.dumps({"type":"count","value":42})),
], schema=StructType([
    StructField("event_id", LongType()),
    StructField("metadata", StringType()),
]))

SAFE_PATH = f"{DATA_DIR}/safe_union"
safe_data.write.format("avro").option("avroSchema", SAFE_SCHEMA).mode("overwrite").save(SAFE_PATH)
df_safe = spark.read.format("avro").load(SAFE_PATH)
print("Pattern 2 — JSON string workaround (works correctly):")
df_safe.show(truncate=False)

## Summary



In [ ]:

print("""
Nullable field patterns:
  Required  : {"name":"col","type":"string"}
  Nullable  : {"name":"col","type":["null","string"],"default":null}
  Optional  : {"name":"col","type":["string","null"],"default":""}

Spark mapping:
  ["null","string"] → StringType(nullable=true)
  ["null","long"]   → LongType(nullable=true)
  ["null","double"] → DoubleType(nullable=true)

Always use ["null","T"] (null first) when the default is null.
Use ["T","null"] only when you want a non-null default.
""")
